### In this notebook I am first creating and storing the images in Images folder. Then I create the mask label for the images and store it in the mask folder
### Then I read the images and masks and first append them to corresponding lists after normalising the pixel values between 0 and 1 and after converting the mask pixels to binary(0 or 1)
### Then the images and mask are split into training testing and validation.
### The Unet model is defined and trained and it outputs the circle_params along with the 

In [3]:
import os
from sklearn.model_selection import train_test_split
import uproot
import numpy as np
import matplotlib.pyplot as plt
import io
from PIL import Image
import cv2
import uproot
from tqdm import tqdm
import tensorflow as tf
from skimage.color import rgb2gray
from keras.models import Model
from keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate
from keras.losses import mean_squared_error
from tensorflow.keras import layers, models
from pathlib import Path


## Load the root file and its various branches

In [ ]:
file = uproot.open("rootfiles/positron.root")
#file = uproot.open("rootfiles/2dilepton.root")

sigtree_pos = file["sig_tree"]
xpixels_pos = sigtree_pos["xPixels"].arrays(library="np")


ypixels_pos = sigtree_pos["yPixels"].arrays(library="np")


geantid_pos=sigtree_pos["GeantID"].arrays(library="np")


xcenter_pos = sigtree_pos["xcenter"].arrays(library="np")
ycenter_pos = sigtree_pos["ycenter"].arrays(library="np")
radius_pos = sigtree_pos["radius"].arrays(library="np")

#Creating the Image dataset
Path("Images/").mkdir(parents=True, exist_ok=True)


## Plot the generated events and save them as images to some directory.

In [ ]:
plt.style.use('default')
fig, ax = plt.subplots()
for i in tqdm(range(0,1000)):
    plt.cla()
    ax.set_xlim(-735, 735)
    ax.set_ylim(-735, 735)
    #geantid = geantid_pos["GeantID"][i]
    xpixel = xpixels_pos["xPixels"][i]
    ypixel = ypixels_pos["yPixels"][i]
    
    plt.scatter(xpixel, ypixel, c = 'black', marker='.', edgecolor='none')
    
    # Set equal scaling
    ax.set_aspect('equal', 'box')
    fig.savefig(f'DileptonImages/positron_SE_{i}.png') #Change this directory or make it locally.
   

In [ ]:
#Create the directory if it does not exist
Path("Mask/").mkdir(parents=True, exist_ok=True)

#Creating the mask images
# Create a figure and axisbinary_crossentropy
plt.style.use('dark_background')
fig, ax = plt.subplots()



for i in tqdm(range(0,1000)):
    # Clear the axis before adding new circles
    plt.cla()
    
    # # Set the limits again, since cla() resets them
    ax.set_xlim(-735, 735)
    ax.set_ylim(-735, 735)
    
    for x, y, r in zip(xcenter_pos["xcenter"][i], ycenter_pos["ycenter"][i], radius_pos["radius"][i]):
        circle = plt.Circle((x, y), r, color='white', fill=False)
        ax.add_patch(circle)
        
        # Set equal scaling
        ax.set_aspect('equal', 'box')
    
    fig.savefig(f'HollowMaskDileptons/mask_{i}.png')

## Load the images and zip them together so that the images are loaded together in the ML algorithm.

In [ ]:
# Define the image and mask directories
image_dir = 'ImagesFGBG'
mask_dir = 'HollowMask'

# Get the list of image and mask files
image_files = sorted([f for f in os.listdir(image_dir)], key=lambda x: int(x.split('.')[0].split('_')[-1]))
mask_files = sorted([f for f in os.listdir(mask_dir)], key=lambda x: int(x.split('.')[0].split('_')[-1]))

In [ ]:
# Create lists to store the image and mask data
images = []

masks = []

# Loop through the image and mask files
for image_file, mask_file in zip(image_files, mask_files):
    # Read the image and mask files
    image_path = os.path.join(image_dir, image_file)
    mask_path = os.path.join(mask_dir, mask_file)
    image = cv2.imread(image_path,cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(mask_path,cv2.IMREAD_GRAYSCALE)

    """
    # Normalize the image pixel values to [0, 1]
    image = image / 255.0
 
    # Convert the mask to a binary mask (0 or 1)
    mask = mask / 255.0
    """

    # Append the image and mask data to the lists
    images.append(image[58:426,144:512])
    masks.append(mask[58:426,144:512])

images = np.array(images)
masks = np.array(masks)

## Split into train-val-test.

In [ ]:
# Split the data into training, validation, and testing sets

train_size = int(0.2 * len(images))

val_size = int(0.1 * len(images))

test_size = len(images) - train_size - val_size

train_images, val_images, test_images = np.split(images, [train_size, train_size + val_size])

train_masks, val_masks, test_masks = np.split(masks, [train_size, train_size + val_size])

## The U-NET algorithm which the ML code is running.

In [ ]:
#For more information on U-NET models, consult https://arxiv.org/pdf/1505.04597
input_shape = (368,368, 1)
inputs = layers.Input(shape=input_shape)

# Encoder
c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
p1 = layers.MaxPooling2D((2, 2), padding = 'same')(c1)

c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
p2 = layers.MaxPooling2D((2, 2), padding = 'same')(c2)

c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c3)
p3 = layers.MaxPooling2D((2, 2), padding = 'same')(c3)

c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(p3)
c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c4)
p4 = layers.MaxPooling2D((2, 2), padding = 'same')(c4)

c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(p4)
c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(c5)

# Decoder
u6 = layers.Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(c5)
u6 = layers.concatenate([u6, c4])
c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(u6)
c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c6)

u7 = layers.Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(c6)
u7 = layers.concatenate([u7, c3])
c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u7)
c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c7)

u8 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c7)
u8 = layers.concatenate([u8, c2])
c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u8)
c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c8)


u9 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c8)
u9 = layers.concatenate([u9, c1])
c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u9)
c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c9)

masked_image = layers.Conv2D(1, (1, 1), name='masked_image')(c9) #1 output which is just the image.

model = models.Model(inputs=[inputs], outputs=[masked_image])

## Running the ML algorithm on the loaded images generated above.

In [ ]:
# Compile the model with separate metrics for each output
model.compile(loss=[tf.keras.losses.MAE], 
              optimizer='adam', 
              metrics={'masked_image': 'accuracy'})

# Assuming train_images, train_masks, val_images, val_masks are already defined
history = model.fit(train_images, train_masks, 
                    batch_size=8, 
                    epochs=5, 
                    validation_data=(val_images, val_masks),
                    shuffle=True)
# Print training and validation loss at each epoch

print("Epoch\tTrain Loss\tVal Loss")

for i, (train_loss, val_loss) in enumerate(zip(history.history['loss'], history.history['val_loss'])):

    print(f"{i+1}\t{train_loss:.4f}\t{val_loss:.4f}")

#Evaluate the model
loss, accuracy = model.evaluate(test_images, test_masks)

print(f'Test loss: {loss:.3f}, Test accuracy: {accuracy:.3f}')

# Print final training and validation metrics
print(f"Final Train Loss: {history.history['loss'][-1]:.4f}")

print(f"Final Val Loss: {history.history['val_loss'][-1]:.4f}")

In [ ]:
predictions = model.predict(test_images)
#Predictions allow to plot the ML models rings.

In [ ]:
masked_image_pred = predictions

c1 = masked_image_pred[0]
#Can plot c1 now using Imshow
plt.imshow(c1)

## Saving the ML algorithm

In [ ]:
#Save the ML model
model.save("model_MAE_5Epoch.h5")

In [ ]:
import keras
#Load the ML model
model = keras.models.load_model("model_MAE_5Epoch.keras")